# 04. Random Forest Regressor

In [ ]:
import sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))
from src.models import get_random_forest, save_model
from src.evaluation import regression_metrics
from src.utils import set_seed, save_json

set_seed(42)
FEAT_DIR = Path('../data_features')
MODEL_DIR = Path('../models')
MODEL_DIR.mkdir(parents=True, exist_ok=True)
RES_DIR = Path('../results/metrics')
RES_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
train = pd.read_parquet(FEAT_DIR / 'train.parquet')
val = pd.read_parquet(FEAT_DIR / 'val.parquet')
test = pd.read_parquet(FEAT_DIR / 'test.parquet')
print(train.shape, val.shape, test.shape)

In [ ]:
TARGET = 'delay_hours'
DROP = [TARGET, 'trip_id', 'load_id', 'driver_id', 'truck_id', 'trailer_id',
        'customer_id', 'route_id', 'dispatch_date']

feature_cols = [c for c in train.columns if c not in DROP
                 and pd.api.types.is_numeric_dtype(train[c])]
print('Features:', len(feature_cols))

X_train, y_train = train[feature_cols], train[TARGET]
X_val, y_val = val[feature_cols], val[TARGET]
X_test, y_test = test[feature_cols], test[TARGET]

## 1. Train baseline RF

In [ ]:
model = get_random_forest()
model.fit(X_train, y_train)

metrics = {
    'train': regression_metrics(y_train, model.predict(X_train)),
    'val':   regression_metrics(y_val,   model.predict(X_val)),
    'test':  regression_metrics(y_test,  model.predict(X_test)),
}
metrics

## 2. Hyperparameter tuning với Optuna

In [ ]:
import optuna
from sklearn.metrics import mean_absolute_error

def objective(trial):
    params = dict(
        n_estimators=trial.suggest_int('n_estimators', 100, 500, step=100),
        max_depth=trial.suggest_int('max_depth', 5, 30),
        min_samples_split=trial.suggest_int('min_samples_split', 2, 20),
        max_features=trial.suggest_categorical('max_features', ['sqrt', 'log2']),
    )
    m = get_random_forest(**params)
    m.fit(X_train, y_train)
    return mean_absolute_error(y_val, m.predict(X_val))

study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=30, show_progress_bar=True)
print('Best params:', study.best_params)
print('Best MAE:', study.best_value)

## 3. Final model + save

In [ ]:
best_model = get_random_forest(**study.best_params)
best_model.fit(X_train, y_train)

final_metrics = {
    'train': regression_metrics(y_train, best_model.predict(X_train)),
    'val':   regression_metrics(y_val,   best_model.predict(X_val)),
    'test':  regression_metrics(y_test,  best_model.predict(X_test)),
    'best_params': study.best_params,
}

save_model(best_model, MODEL_DIR / 'random_forest.pkl')
save_json(final_metrics, RES_DIR / 'random_forest.json')
final_metrics